# BRISC 2025 — Round 4 addendum: reproducible statistics from what is already on Drive

**Purpose.** Every number the revised manuscript needs that can be computed from saved artifacts, with no training.
Reads `MyDrive/brisc_runs` (per-run `pred_test.npz`, `done.json`, `results_bundle.zip`) and `MyDrive/brisc_reuse`
(Cell 32 explanation tables). Writes everything to `MyDrive/brisc_round4/` and downloads one zip.

**How to run.** New Colab notebook, any runtime (CPU is enough) → upload this file → *Runtime ▸ Run all* → allow Drive.
Expected time **5–10 min** in total (A0 2–5 min, the rest 1–3 min each). Each cell ends with `CELL Ax CLEAN.` or names its failures.

**Send back.** `brisc_round4_results.zip` (downloads at the end) **and** this notebook with its outputs (*File ▸ Download ▸ .ipynb*).

| Cell | What it settles in the manuscript |
|---|---|
| A0 | Run-trace table (run-id, config signature, epoch, GPU, minutes) for every run; one per-image table for the data deposit |
| A1 | All 32 pre-specified rows + 5 negative controls recomputed **in code**, with Satterthwaite CIs, MDEs, Holm/BH; round-3 rows (other networks, earlier design) **exact** instead of summary-level; comparison with the manuscript |
| A2 | Balance of the two randomisation halves; a randomisation test of the crossover that needs no t approximation |
| A3 | Metrics without the no-tumour shortcut (3 tumour classes); the crossover evaluations with CIs; exposure effect by class; headroom closed (after B1) |
| A4 | Every explanation result per image with corrected CIs: all four methods, by class, H1–H4, other models |
| A5 | Orientation of every traced image relative to its figshare source |
| A9 | Summary + zip |
| B1 | **Copy into the main notebook, after Cell 7** (it skips itself here): per-image twin-mask Dice; train–test copy search in the 6 rotated orientations |


In [ ]:
# =============================================================================
#  CELL A0  |  Setup, Drive inventory, run-trace table, per-image predictions
#              Fresh runtime, CPU is enough. Reads only what is on Drive. ~2-5 min.
# =============================================================================
import os, json, re, time, zipfile, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats

T0 = time.time()
MOCK = os.environ.get("BRISC_DRIVE")                 # set only when this notebook is tested offline
if MOCK:
    DRIVE = Path(MOCK)
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive")
RUNS, REUSE = DRIVE / "brisc_runs", DRIVE / "brisc_reuse"
OUT = DRIVE / "brisc_round4"; OUT.mkdir(parents=True, exist_ok=True)
BUNDLE = RUNS / "results_bundle.zip"

N_BOOT, BOOT_SEED = 10_000, 20261015
CLASS = {0: "no_tumor", 1: "glioma", 2: "meningioma", 3: "pituitary"}
LINKED = ["copy_plain", "copy_mirror", "patient_only", "related"]
CHECKS = {}

def banner(t):
    print("\n" + "=" * 104 + f"\n  {t}\n" + "=" * 104)

def done(cell):
    bad = [k for k, v in CHECKS.items() if k.startswith(cell) and not v]
    for k, v in CHECKS.items():
        if k.startswith(cell):
            print(f"  [{'PASS' if v else 'FAIL'}] {k}")
    print(f"CELL {cell} CLEAN." if not bad else f"CELL {cell} HAS {len(bad)} FAILURE(S) - send the output as it is.")

# ----- A0.1 manifests --------------------------------------------------------------------------------------------
banner("A0.1  MANIFESTS FROM results_bundle.zip")
assert RUNS.exists(), f"{RUNS} not found: mount the Drive account that holds brisc_runs"
assert BUNDLE.exists(), f"{BUNDLE} not found"
with zipfile.ZipFile(BUNDLE) as z:
    names = set(z.namelist())
    miss = [n for n in ("manifest_cell07.csv", "cell11_sets.csv") if n not in names]
    assert not miss, f"missing inside the bundle: {miss}"
    man = pd.read_csv(z.open("manifest_cell07.csv"), low_memory=False)
    sets = pd.read_csv(z.open("cell11_sets.csv"), low_memory=False)
CHECKS["A0 the two manifests list the same images in the same order"] = bool(
    len(man) == len(sets) and (man.image_id.to_numpy() == sets.image_id.to_numpy()).all())
man["in_Fm"] = sets["in_Fm"].to_numpy() if "in_Fm" in sets else False
man["has_mask"] = man["has_mask"].astype(str).str.lower().isin(["true", "1"])
man["stratum"] = man["stratum"].fillna("").astype(str)
for c_ in ("traced",):
    if c_ in man:
        man[c_] = man[c_].astype(str).str.lower().isin(["true", "1"])
TEST = man[man.split == "test"].copy()
TROWS = TEST.index.to_numpy(); n = len(TROWS); POS = {int(r): k for k, r in enumerate(TROWS)}
lab = TEST.label_idx.to_numpy().astype(int); half = TEST.half.to_numpy().astype(int)
strat = TEST.stratum.to_numpy(); tum = TEST.has_mask.to_numpy()
print(f"{len(man)} images; test {n} ({int(tum.sum())} with a tumour mask)")
print(TEST.stratum.value_counts().to_string())
CHECKS["A0 1,000 test images, 860 with a mask"] = bool(n == 1000 and tum.sum() == 860)

# ----- A0.2 run inventory and trace table -----------------------------------------------------------------------
banner("A0.2  RUN INVENTORY (every folder in brisc_runs)  ->  runs_trace.csv")
def split_tag(t):
    m = re.match(r"^(.*)_s(\d+)$", t)
    return (m.group(1), int(m.group(2))) if m else (t, None)
rows_ = []
for d in sorted(p for p in RUNS.iterdir() if p.is_dir()):
    arm, seed = split_tag(d.name)
    dj = json.loads((d / "done.json").read_text()) if (d / "done.json").exists() else {}
    rows_.append({"tag": d.name, "arm": arm, "seed": seed, "pred_test": (d / "pred_test.npz").exists(),
                  "pred_test_last": (d / "pred_test_last.npz").exists(), "best_pt": (d / "best.pt").exists(),
                  "task": dj.get("task"), "kind": dj.get("kind", "brisc"), "set": dj.get("set"), "cfg_sig": dj.get("cfg_sig"),
                  "best_epoch": dj.get("best_epoch"), "val_score": dj.get("val_score"), "train_images": dj.get("train_images"),
                  "minutes": dj.get("minutes"), "gpu": dj.get("gpu"), "torch": dj.get("torch"), "cache_md5": dj.get("cache_md5")})
RT = pd.DataFrame(rows_)
RT.to_csv(OUT / "runs_trace.csv", index=False)
with pd.option_context("display.max_rows", 300, "display.width", 220):
    print(RT[["tag", "pred_test", "best_pt", "task", "kind", "set", "best_epoch", "minutes", "gpu", "cfg_sig"]].to_string(index=False))
print(f"\n{int(RT.pred_test.sum())} runs with test predictions; {int(RT.best_pt.sum())} with a checkpoint on Drive")

# ----- A0.3 per-image predictions -------------------------------------------------------------------------------
banner("A0.3  PER-IMAGE TEST PREDICTIONS OF EVERY RUN")
P = {}                                     # P[arm][seed] = dict of per-image vectors in test order
TASK = {}
for _, r in RT[RT.pred_test].iterrows():
    z = np.load(RUNS / r.tag / "pred_test.npz")
    rows_z = z["row"].astype(int)
    if sorted(rows_z.tolist()) != sorted(TROWS.tolist()):
        print(f"  SKIP {r.tag}: its test rows differ from the manifest"); continue
    o = np.array([POS[int(x)] for x in rows_z])
    prob = np.full((n, 4), np.nan); prob[o] = z["prob"]
    dice = np.full(n, np.nan); dice[o] = z["dice"]
    gpx = np.zeros(n); gpx[o] = z["gt_px"]; ppx = np.zeros(n); ppx[o] = z["pred_px"]
    ok_p = np.isfinite(prob).all(1)
    pred = np.where(ok_p, np.nan_to_num(prob).argmax(1), -1)
    p_true = np.where(ok_p, prob[np.arange(n), lab], np.nan)
    rec = {"pred": pred, "p_true": p_true, "logp": np.log(np.clip(p_true, 1e-12, 1.0)),
           "correct": np.where(ok_p, (pred == lab).astype(float), np.nan),
           "dice": np.where(gpx > 0, dice, np.nan), "pred_px": ppx, "gt_px": gpx, "prob": prob}
    P.setdefault(r.arm, {})[int(r.seed)] = rec
    t_ = r.task or {"effb0": "cls", "swint": "cls", "unet": "seg"}.get(str(r.kind), "both")
    if not np.isfinite(rec["dice"]).any():
        t_ = "cls"                                   # no segmentation output stored
    elif not ok_p.any():
        t_ = "seg"                                   # no class probabilities stored
    TASK[r.arm] = t_
CHECKS["A0 Dice is defined exactly on the tumour images (segmenting arms)"] = all(
    np.array_equal(np.isfinite(v["dice"]), tum) for a, d_ in P.items() for v in d_.values() if TASK[a] in ("both", "seg"))
print(pd.DataFrame([{"arm": a, "seeds": sorted(d_), "task": TASK[a]} for a, d_ in sorted(P.items())]).to_string(index=False))

# ----- A0.4 one long per-image table (for the data deposit) ------------------------------------------------------
long_ = []
for a, d_ in P.items():
    for s_, v in d_.items():
        long_.append(pd.DataFrame({"arm": a, "seed": s_, "image_id": TEST.image_id.to_numpy(), "label": TEST.label.to_numpy(),
                                   "stratum": strat, "half": half, "group": TEST.grp.to_numpy(), "pred": v["pred"],
                                   "p_true": v["p_true"], "dice": v["dice"], "pred_px": v["pred_px"].astype(int)}))
PIMG = pd.concat(long_, ignore_index=True)
PIMG.to_csv(OUT / "per_image_all_runs.csv.gz", index=False)
print(f"\nper_image_all_runs.csv.gz: {len(PIMG)} rows ({PIMG.arm.nunique()} arms)")

# ----- A0.5 explanation results cached by Cell 32 ---------------------------------------------------------------
banner("A0.5  CELL 32 EXPLANATION TABLES IN brisc_reuse")
X32 = {}
if REUSE.exists():
    found = {}
    for p_ in REUSE.glob("cell32_*.pkl"):
        m = re.match(r"^cell32_(.+)_([0-9a-f]{16})\.pkl$", p_.name)
        if m and (m.group(1) not in found or p_.stat().st_mtime > found[m.group(1)].stat().st_mtime):
            found[m.group(1)] = p_
    for tag, p_ in sorted(found.items()):
        X32[tag] = pd.read_pickle(p_)
print(f"{len(X32)} explanation tables: {sorted(X32)}")
CHECKS["A0 explanation tables found for the clean replicate (3 seeds)"] = sum(t.startswith("B_flip_w_s") for t in X32) == 3
print(f"\nA0 finished in {time.time() - T0:.0f}s")
done("A0")


In [ ]:
# =============================================================================
#  CELL A1  |  The corrected analysis, reproducibly: every pre-specified row of
#              Table S1 recomputed from per-image predictions (round-3 rows now
#              exact, not summary-level), checked against the manuscript. ~3-6 min.
#  Method (as described in Section 2.5 of the manuscript, reconstructed here):
#    variance = group-bootstrap variance of the seed-averaged estimate (groups of related test images resampled within
#    each randomisation half) + between-seed variance / k; Satterthwaite df from (groups - 1) and (k - 1); two-sided t
#    test; MDE = (z_0.975 + z_0.80) x SE; Holm over the 5 primary endpoints; Benjamini-Hochberg over the 27 secondary.
# =============================================================================
assert "P" in globals() and "TEST" in globals(), "Run Cell A0 first."
T1 = time.time()

# ----- A1.1 bootstrap weights ----------------------------------------------------------------------------------
ug, ginv = np.unique(TEST.grp.to_numpy(), return_inverse=True)
half_g = pd.Series(half).groupby(ginv).first().to_numpy()
CHECKS["A1 no group spans two randomisation halves"] = bool((pd.Series(half).groupby(ginv).nunique() == 1).all())
rng = np.random.default_rng(BOOT_SEED)
Wg = np.zeros((N_BOOT, len(ug)), np.float32)
for h_ in np.unique(half_g):
    ix = np.where(half_g == h_)[0]
    Wg[:, ix] = rng.multinomial(len(ix), np.full(len(ix), 1.0 / len(ix)), size=N_BOOT)
W = Wg[:, ginv]; W1 = np.ones((1, n), np.float32)
print(f"{len(ug)} groups of related test images; {N_BOOT} bootstrap replicates (seed {BOOT_SEED})")

def wmean(x, Wm, mask=None):
    ok = np.isfinite(x) & (np.ones(n, bool) if mask is None else mask)
    xs = np.where(ok, x, 0.0); w_ = Wm @ ok.astype(np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(w_ > 0, (Wm @ xs) / np.where(w_ > 0, w_, 1.0), np.nan)

def cls_all(pred, Wm):
    ok = pred >= 0
    E = np.zeros((n, 16), np.float32); E[np.where(ok)[0], lab[ok] * 4 + pred[ok]] = 1
    cm = (Wm @ E).reshape(-1, 4, 4); tp = np.diagonal(cm, axis1=1, axis2=2)
    fp, fn = cm.sum(1) - tp, cm.sum(2) - tp
    f1 = np.where(tp > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1e-9), 0.0)
    rec = tp / np.maximum(cm.sum(2), 1e-9)
    return {"macro_f1": f1.mean(1), "accuracy": tp.sum(1) / np.maximum(cm.sum((1, 2)), 1e-9), "balanced_acc": rec.mean(1),
            "f1_tumour3": f1[:, 1:].mean(1), "recall_tumour3": rec[:, 1:].mean(1)}

_cache = {}
def metric(arm, seed, name, Wm, wkey):
    """name: macro_f1 | accuracy | balanced_acc | f1_tumour3 | recall_tumour3 | dice | logp | correct"""
    k_ = (arm, seed, name, wkey)
    if k_ not in _cache:
        v = P[arm][seed]
        if name in ("dice", "logp", "correct"):
            _cache[k_] = wmean(v[name], Wm, tum if name == "dice" else None)
        else:
            for nm, val in cls_all(v["pred"], Wm).items():
                _cache[(arm, seed, nm, wkey)] = val
    return _cache[k_]

Z_MDE = stats.norm.ppf(0.975) + stats.norm.ppf(0.80)
def combine(est, boot, per_seed, n_groups):
    """Satterthwaite combination of group-bootstrap and between-seed variance."""
    boot = np.asarray(boot)[np.isfinite(boot)]; per_seed = np.asarray(per_seed, float)
    a = float(np.var(boot, ddof=1)); k = len(per_seed)
    b = float(np.var(per_seed, ddof=1) / k) if k > 1 else 0.0
    se = np.sqrt(a + b); dfg, dfs = max(n_groups - 1, 1), max(k - 1, 1)
    den = a ** 2 / dfg + (b ** 2 / dfs if b > 0 else 0.0)
    df_ = (a + b) ** 2 / den if den > 0 else np.inf
    tc = stats.t.ppf(0.975, df_)
    lo_b, hi_b = np.percentile(boot, [2.5, 97.5])
    return {"estimate": est, "se": se, "df": df_, "lo": est - tc * se, "hi": est + tc * se,
            "p": float(2 * stats.t.sf(abs(est) / se, df_)) if se > 0 else np.nan, "mde": Z_MDE * se,
            "boot_lo": lo_b, "boot_hi": hi_b, "var_group": a, "var_seed": b, "boot_valid": float(len(boot) / N_BOOT),
            "per_seed": " ".join(f"{v:+.4f}" for v in per_seed), "k": k, "groups": n_groups,
            "rule_original": ("supported" if (lo_b > 0 or hi_b < 0) and k > 1 and est != 0
                              and all(np.sign(v) == np.sign(est) for v in per_seed) else "not supported")}

def ngroups(mask):
    return int(len(np.unique(ginv[mask])))

def arm_diff(a_, b_, name):
    seeds = sorted(set(P.get(a_, {})) & set(P.get(b_, {})))
    if len(seeds) == 0:
        return None
    per = [float(metric(a_, s, name, W1, "W1")[0] - metric(b_, s, name, W1, "W1")[0]) for s in seeds]
    boot = np.mean([metric(a_, s, name, W, "W") - metric(b_, s, name, W, "W") for s in seeds], 0)
    mask = tum if name == "dice" else np.ones(n, bool)
    r = combine(float(np.mean(per)), boot, per, ngroups(mask)); r["images"] = int(mask.sum())
    return r

def field_mat(arm, field):
    seeds = sorted(P.get(arm, {}))
    return (np.stack([P[arm][s][field] for s in seeds]) if seeds else None), seeds

def half_balanced(x, Wm, m_):
    h1, h2 = m_ & (half == 1), m_ & (half == 2)
    if h1.any() and h2.any():
        return 0.5 * (wmean(x, Wm, h1) + wmean(x, Wm, h2)), True
    return wmean(x, Wm, m_), False

def nanmean0(M):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(M, 0)

def xo_delta(arm1, arm2, field):
    M1, s1 = field_mat(arm1, field); M2, s2 = field_mat(arm2, field)
    if M1 is None or M2 is None:
        return None, []
    seeds = sorted(set(s1) & set(s2))
    D = np.stack([np.where(half == 1, M1[s1.index(s)] - M2[s2.index(s)],
                           np.where(half == 2, M2[s2.index(s)] - M1[s1.index(s)], M1[s1.index(s)] - M2[s2.index(s)]))
                  for s in seeds])
    return D, seeds

def crossover(arm1, arm2, field, stratum):
    D, seeds = xo_delta(arm1, arm2, field)
    if D is None:
        return None
    Dm = nanmean0(D); m_ = (strat == stratum) & np.isfinite(Dm)
    est, both = half_balanced(Dm, W1, m_); boot = half_balanced(Dm, W, m_)[0]
    per = [float(half_balanced(np.nan_to_num(D[k]), W1, m_)[0][0]) for k in range(len(seeds))]
    r = combine(float(est[0]), boot, per, ngroups(m_)); r["images"] = int(m_.sum()); r["both_halves"] = both
    return r

def interaction(field):
    Df, sf = xo_delta("X1_flip", "X2_flip", field); Dn, sn = xo_delta("X1_noflip", "X2_noflip", field)
    if Df is None or Dn is None:
        return None
    common = sorted(set(sf) & set(sn))
    Df = Df[[sf.index(s) for s in common]]; Dn = Dn[[sn.index(s) for s in common]]
    Dd = nanmean0(Df) - nanmean0(Dn)
    mm, mp = (strat == "copy_mirror") & np.isfinite(Dd), (strat == "copy_plain") & np.isfinite(Dd)
    est = half_balanced(Dd, W1, mm)[0] - half_balanced(Dd, W1, mp)[0]
    boot = half_balanced(Dd, W, mm)[0] - half_balanced(Dd, W, mp)[0]
    per = []
    for k in range(len(common)):
        dk = np.nan_to_num(Df[k] - Dn[k])
        per.append(float(half_balanced(dk, W1, mm)[0][0] - half_balanced(dk, W1, mp)[0][0]))
    r = combine(float(est[0]), boot, per, ngroups(mm | mp)); r["images"] = int((mm | mp).sum())
    return r

# ----- A1.2 the rows of Table S1 (frozen 13 Oct 2026), with the manuscript's corrected values ----------------------
# key: (level, label, endpoint, how, args, paper lo, paper hi, paper MDE, paper adjusted p or q)
XO = ("X1_flip", "X2_flip"); XN = ("X1_noflip", "X2_noflip")
ROWS = [
    ("P1", "primary", "Crossover, plain copies", "Dice", "xo", (*XO, "dice", "copy_plain"), .0155, .0759, .0410, .0182),
    ("P2", "primary", "Crossover, mirror-image copies", "Dice", "xo", (*XO, "dice", "copy_mirror"), .0077, .0303, .0153, .0103),
    ("P3", "primary", "Crossover, plain copies", "Log probability", "xo", (*XO, "logp", "copy_plain"), .0338, .6229, .4033, .0603),
    ("P4", "primary", "Crossover, mirror-image copies", "Log probability", "xo", (*XO, "logp", "copy_mirror"), .0267, .5390, .3636, .0603),
    ("P5", "primary", "Size-matched minus clean", "Dice", "diff", ("Fm_flip", "B_flip", "dice"), .0062, .0366, .0212, .0206),
    ("S01", "secondary", "Standard minus clean", "Macro-F1", "diff", ("full_flip", "B_flip", "macro_f1"), .0371, .0777, .0282, .0009),
    ("S02", "secondary", "Standard minus clean", "Dice", "diff", ("full_flip", "B_flip", "dice"), .0766, .1088, .0229, .0009),
    ("S03", "secondary", "Standard minus clean", "Log probability", "diff", ("full_flip", "B_flip", "logp"), .1803, .4424, .1845, .0009),
    ("S04", "secondary", "Crossover, same-patient slices", "Dice", "xo", (*XO, "dice", "patient_only"), -.0107, .0408, .0355, .4411),
    ("S05", "secondary", "Crossover, same-patient slices", "Log probability", "xo", (*XO, "logp", "patient_only"), -.0307, .3228, .2466, .2583),
    ("S06", "secondary", "Crossover, near-duplicates", "Dice", "xo", (*XO, "dice", "related"), -.0267, .0336, .0399, .8428),
    ("S07", "secondary", "Crossover, near-duplicates", "Log probability", "xo", (*XO, "logp", "related"), -.2047, .4255, .3736, .5592),
    ("S08", "secondary", "Crossover, plain copies", "Share correct", "xo", (*XO, "correct", "copy_plain"), .0129, .1008, .0619, .0459),
    ("S09", "secondary", "Crossover, mirror-image copies", "Share correct", "xo", (*XO, "correct", "copy_mirror"), .0115, .0671, .0394, .0262),
    ("S10", "secondary", "Crossover, same-patient slices", "Share correct", "xo", (*XO, "correct", "patient_only"), -.0068, .0757, .0542, .2583),
    ("S11", "secondary", "Flip x mirror interaction", "Dice", "inter", ("dice",), -.0408, .0473, .0445, .8478),
    ("S12", "secondary", "Flip x mirror interaction", "Log probability", "inter", ("logp",), -.0805, .2870, .2558, .4699),
    ("S13", "secondary", "Crossover without flip, plain copies", "Dice", "xo", (*XN, "dice", "copy_plain"), .0168, .0734, .0386, .0156),
    ("S14", "secondary", "Crossover without flip, mirror-image copies", "Dice", "xo", (*XN, "dice", "copy_mirror"), .0055, .0249, .0135, .0156),
    ("S15", "secondary", "FiLM minus shuffled-plane control", "Macro-F1", "diff", ("B_flip_film", "B_flip_placebo", "macro_f1"), -.0161, .0382, .0256, .4699),
    ("S16", "secondary", "FiLM minus shuffled-plane control", "Dice", "diff", ("B_flip_film", "B_flip_placebo", "dice"), -.0033, .0123, .0110, .4699),
    ("S17", "secondary", "FiLM minus no conditioning", "Macro-F1", "diff", ("B_flip_film", "B_flip", "macro_f1"), -.0139, .0321, .0220, .4699),
    ("S18", "secondary", "FiLM minus no conditioning", "Dice", "diff", ("B_flip_film", "B_flip", "dice"), -.0069, .0217, .0158, .4411),
    ("S19", "secondary", "Shuffled-plane control minus no conditioning", "Macro-F1", "diff", ("B_flip_placebo", "B_flip", "macro_f1"), -.0108, .0070, .0125, .7973),
    ("S20", "secondary", "Shuffled-plane control minus no conditioning", "Dice", "diff", ("B_flip_placebo", "B_flip", "dice"), -.0167, .0224, .0190, .7851),
    ("S21", "secondary", "Joint minus segmentation-only (clean set)", "Dice", "diff", ("B_flip", "B_flip_seg", "dice"), -.0297, -.0041, .0159, .0506),
    ("S22", "secondary", "Joint minus classification-only (clean set)", "Macro-F1", "diff", ("B_flip", "B_flip_cls", "macro_f1"), -.0355, .0657, .0412, .5252),
    ("S23", "secondary", "Joint minus segmentation-only (standard set)", "Dice", "diff", ("full_flip", "full_flip_seg", "dice"), -.0134, .0083, .0105, .6583),
    ("S24", "secondary", "Joint minus classification-only (standard set)", "Macro-F1", "diff", ("full_flip", "full_flip_cls", "macro_f1"), -.0033, .0226, .0127, .2583),
    ("S25", "secondary", "Size-matched minus clean", "Macro-F1", "diff", ("Fm_flip", "B_flip", "macro_f1"), -.0192, .0305, .0283, .7020),
    ("S26", "secondary", "Earlier design minus our network (clean set)", "Macro-F1", "diff", ("B_flip_p1", "B_flip", "macro_f1"), -.0188, .0034, .0153, .3515),
    ("S27", "secondary", "Earlier design minus our network (clean set)", "Dice", "diff", ("B_flip_p1", "B_flip", "dice"), -.0559, .0146, .0229, .2874),
    ("N1", "negative control", "Crossover models, unlinked images", "Dice", "xo", (*XO, "dice", "unlinked"), -.0687, .0630, .0757, None),
    ("N2", "negative control", "Crossover models, unlinked images", "Log probability", "xo", (*XO, "logp", "unlinked"), -.0857, .0543, .0940, None),
    ("N3", "negative control", "Crossover models, unlinked images", "Share correct", "xo", (*XO, "correct", "unlinked"), -.0245, .0144, .0245, None),
    ("N4", "negative control", "Crossover without flip, unlinked images", "Dice", "xo", (*XN, "dice", "unlinked"), -.0117, .0506, .0423, None),
    ("N5", "negative control", "Crossover without flip, unlinked images", "Log probability", "xo", (*XN, "logp", "unlinked"), -.0342, .1447, .1262, None),
    # round 3 (12 Oct 2026): in the manuscript these kept their original rule only; now estimated per image
    ("R1", "round 3", "EfficientNet-B0: standard minus clean", "Macro-F1", "diff", ("full_flip_effb0", "B_flip_effb0", "macro_f1"), None, None, None, None),
    ("R2", "round 3", "Swin-T: standard minus clean", "Macro-F1", "diff", ("full_flip_swint", "B_flip_swint", "macro_f1"), None, None, None, None),
    ("R3", "round 3", "U-Net: standard minus clean", "Dice", "diff", ("full_flip_unet", "B_flip_unet", "dice"), None, None, None, None),
    ("R4", "round 3", "Our network, classification only: standard minus clean", "Macro-F1", "diff", ("full_flip_cls", "B_flip_cls", "macro_f1"), None, None, None, None),
    ("R5", "round 3", "Our network, segmentation only: standard minus clean", "Dice", "diff", ("full_flip_seg", "B_flip_seg", "dice"), None, None, None, None),
]

banner("A1.2  EVERY ROW, RECOMPUTED")
out_ = []
for key, level, comp, endp, how, args, plo, phi, pmde, padj in ROWS:
    r = (arm_diff(*args) if how == "diff" else crossover(*args) if how == "xo" else interaction(*args))
    if r is None:
        print(f"  {key}: not estimable (missing arm among {args})"); continue
    out_.append({"key": key, "level": level, "comparison": comp, "endpoint": endp, **r,
                 "paper_lo": plo, "paper_hi": phi, "paper_mde": pmde, "paper_adj": padj})
A1 = pd.DataFrame(out_)
A1["adjusted"] = np.nan
for lev, fn in (("primary", "holm"), ("secondary", "bh"), ("round 3", "bh")):
    m_ = A1.level == lev
    p_ = A1.loc[m_, "p"].to_numpy()
    if len(p_) == 0:
        continue
    if fn == "holm":
        o = np.argsort(p_); adj = np.empty(len(p_)); run = 0.0
        for i, j in enumerate(o):
            run = max(run, (len(p_) - i) * p_[j]); adj[j] = min(1.0, run)
    else:
        o = np.argsort(p_); q = p_[o] * len(p_) / np.arange(1, len(p_) + 1)
        q = np.minimum.accumulate(q[::-1])[::-1]; adj = np.empty(len(p_)); adj[o] = np.minimum(q, 1.0)
    A1.loc[m_, "adjusted"] = adj
CHECKS["A1 the frozen families have 5 primary and 27 secondary rows"] = bool(
    (A1.level == "primary").sum() == 5 and (A1.level == "secondary").sum() == 27)
A1["verdict"] = np.where(A1.level == "negative control", "negative control, no test",
                         np.where(A1.adjusted < 0.05, "significant", "not significant"))
A1["d_lo"] = A1.lo - A1.paper_lo; A1["d_hi"] = A1.hi - A1.paper_hi
A1["paper_verdict"] = np.where(A1.paper_adj.isna(), None, np.where(A1.paper_adj < 0.05, "significant", "not significant"))
A1.to_csv(OUT / "A1_corrected_all_rows.csv", index=False)
show = A1[["key", "comparison", "endpoint", "images", "groups", "estimate", "lo", "hi", "df", "mde", "p", "adjusted", "verdict",
           "per_seed", "paper_lo", "paper_hi", "d_lo", "d_hi"]].copy()
with pd.option_context("display.max_rows", 200, "display.width", 260, "display.max_colwidth", 46):
    print(show.round(4).to_string(index=False))

# ----- A1.3 agreement with the manuscript --------------------------------------------------------------------------
banner("A1.3  AGREEMENT WITH THE MANUSCRIPT'S CORRECTED VALUES (13 Oct 2026)")
cmp_ = A1[A1.paper_lo.notna()].copy()
cmp_["max_abs_delta"] = np.maximum(cmp_.d_lo.abs(), cmp_.d_hi.abs())
same_v = cmp_[cmp_.paper_verdict.notna()]
n_flip = int((same_v.verdict != same_v.paper_verdict).sum())
print(f"rows compared: {len(cmp_)}; largest |difference| in a CI bound: {cmp_.max_abs_delta.max():.4f} "
      f"(row {cmp_.loc[cmp_.max_abs_delta.idxmax(), 'key']}); median {cmp_.max_abs_delta.median():.4f}")
print(f"verdicts that differ from the manuscript: {n_flip}")
if n_flip:
    print(same_v.loc[same_v.verdict != same_v.paper_verdict, ["key", "comparison", "endpoint", "adjusted", "paper_adj"]].to_string(index=False))
print("Rows S26/S27 were 'approximate' in the manuscript (summary values); the new values are exact and replace them.")
CHECKS["A1 primary CI bounds within 0.005 of the manuscript"] = bool(
    cmp_[cmp_.level == "primary"].max_abs_delta.max() < 0.005)
print(f"\nA1 finished in {time.time() - T1:.0f}s")
done("A1")


In [ ]:
# =============================================================================
#  CELL A2  |  Was the randomisation balanced, and does a randomisation test agree
#              with the t-based result?  (CPU, ~1-3 min)
#  (a) Balance: the two halves compared on class, plane, image-size family, figshare tracing and storage orientation.
#      A source or geometry imbalance between halves is the one route by which a generic difference between the two
#      crossover models could leak into the equal-weighted estimator.
#  (b) Conditional randomisation test: the group-to-half assignment is redrawn with the algorithm of Cell 6
#      (groups by size, each to the half with the lighter load of its class, ties at random); the estimator is recomputed
#      on the fixed per-image differences X1 - X2. Conditional on the trained models; no bootstrap, no t approximation.
# =============================================================================
assert "crossover" in globals(), "Run Cells A0 and A1 first."
T2 = time.time()
N_PERM = 5000

# ----- A2.1 balance ------------------------------------------------------------------------------------------------
banner("A2.1  BALANCE OF THE TWO RANDOMISATION HALVES")
def smd(p1, p2):
    s_ = np.sqrt((p1 * (1 - p1) + p2 * (1 - p2)) / 2)
    return (p1 - p2) / s_ if s_ > 0 else 0.0
tr_c = man[man.role.isin(["C1", "C2"])].copy(); tr_c["h"] = np.where(tr_c.role == "C1", 1, 2)
te_l = TEST[TEST.half.isin([1, 2])].copy(); te_l["h"] = te_l.half
bal = []
for who, d_ in (("training images (C1 vs C2)", tr_c), ("linked test images (half 1 vs half 2)", te_l)):
    feats = {"class": d_.label, "plane": d_.plane if "plane" in d_ else None,
             "size family": d_.size_family if "size_family" in d_ else None,
             "traced to figshare": d_.traced.map({True: "traced", False: "not traced"}) if "traced" in d_ else None,
             "stored mirrored vs source": (d_.fig_variant.fillna("untraced").astype(str).str.contains("mirror")
                                           .map({True: "mirrored", False: "not mirrored"}) if "fig_variant" in d_ else None)}
    if who.startswith("linked"):
        feats["kind of relative"] = d_.stratum
    for fname, col in feats.items():
        if col is None:
            continue
        for lev in sorted(col.astype(str).unique()):
            x1 = float((col.astype(str)[d_.h == 1] == lev).mean()); x2 = float((col.astype(str)[d_.h == 2] == lev).mean())
            bal.append({"set": who, "feature": fname, "level": lev, "n_half1": int(((col.astype(str) == lev) & (d_.h == 1)).sum()),
                        "n_half2": int(((col.astype(str) == lev) & (d_.h == 2)).sum()), "share_half1": x1, "share_half2": x2,
                        "SMD": smd(x1, x2)})
BAL = pd.DataFrame(bal); BAL.to_csv(OUT / "A2_balance.csv", index=False)
with pd.option_context("display.max_rows", 200, "display.width", 200):
    print(BAL.round(3).to_string(index=False))
worst = BAL.loc[BAL.SMD.abs().idxmax()]
print(f"\nlargest |SMD|: {abs(worst.SMD):.3f} ({worst.set}; {worst.feature} = {worst.level}); "
      "|SMD| < 0.1 is the usual threshold for negligible imbalance")

# ----- A2.2 conditional randomisation test --------------------------------------------------------------------------
banner(f"A2.2  CONDITIONAL RANDOMISATION TEST OF THE CROSSOVER ({N_PERM} re-randomisations)")
cg = tr_c.groupby("grp").agg(n=("label", "size"), cls=("label", lambda s: s.value_counts().index[0]))
g_ids, g_n, g_cls = cg.index.to_numpy(), cg.n.to_numpy(), cg.cls.to_numpy()
obs_h = tr_c.groupby("grp").h.first().reindex(g_ids).to_numpy()
cls_list = sorted(set(g_cls)); cls_i = np.array([cls_list.index(c) for c in g_cls])
load_obs = {(c, h): int(g_n[(g_cls == c) & (obs_h == h)].sum()) for c in cls_list for h in (1, 2)}
print("observed training-image load by class and half:", {f"{c}|{h}": v for (c, h), v in load_obs.items()})
te_gpos = pd.Series(np.arange(len(g_ids)), index=g_ids)
te_pos = TEST.grp.map(te_gpos).to_numpy()                     # position of each test image's group, NaN if unlinked
CHECKS["A2 every linked test image belongs to a group with training relatives"] = bool(
    np.isfinite(te_pos[half > 0]).all())
ht_obs = np.zeros(n, int); lk = np.isfinite(te_pos); ht_obs[lk] = obs_h[te_pos[lk].astype(int)]
CHECKS["A2 the halves of the test images equal the halves of their training groups"] = bool(np.array_equal(ht_obs, half))
rngp = np.random.default_rng(BOOT_SEED + 1)

def redraw():
    perm = rngp.permutation(len(g_ids))
    order = perm[np.argsort(-g_n[perm], kind="stable")]
    load = np.zeros((len(cls_list), 2)); h = np.zeros(len(g_ids), int)
    for j in order:
        c = cls_i[j]; a1, a2 = load[c, 0], load[c, 1]
        hj = 1 if a1 < a2 else 2 if a2 < a1 else int(rngp.integers(1, 3))
        h[j] = hj; load[c, hj - 1] += g_n[j]
    return h

TESTS = [("copy_plain", "dice"), ("copy_mirror", "dice"), ("copy_plain", "logp"), ("copy_mirror", "logp"),
         ("patient_only", "dice"), ("copy_plain", "correct"), ("copy_mirror", "correct")]
Draw = {}
for st, f in TESTS:
    M1, s1 = field_mat("X1_flip", f); M2, s2 = field_mat("X2_flip", f)
    common = sorted(set(s1) & set(s2))
    Draw[(st, f)] = nanmean0(np.stack([M1[s1.index(s)] - M2[s2.index(s)] for s in common]))
def stat_of(hvec, st, f):
    D = Draw[(st, f)]; m_ = (strat == st) & np.isfinite(D) & np.isfinite(te_pos)
    ht = np.zeros(n, int); ht[m_] = hvec[te_pos[m_].astype(int)]
    a, b = m_ & (ht == 1), m_ & (ht == 2)
    return 0.5 * (D[a].mean() - D[b].mean()) if a.any() and b.any() else np.nan
obs = {k: stat_of(obs_h, *k) for k in TESTS}
null = {k: [] for k in TESTS}
for _ in range(N_PERM):
    hv = redraw()
    for k in TESTS:
        null[k].append(stat_of(hv, *k))
rows_ = []
for k in TESTS:
    nl = np.asarray(null[k]); nl = nl[np.isfinite(nl)]
    rows_.append({"stratum": k[0], "endpoint": k[1], "estimate": obs[k], "null_sd": nl.std(ddof=1),
                  "p_randomisation": (1 + (np.abs(nl) >= abs(obs[k])).sum()) / (1 + len(nl)), "draws": len(nl)})
A2R = pd.DataFrame(rows_)
A2R.to_csv(OUT / "A2_randomisation_test.csv", index=False)
print(A2R.round(4).to_string(index=False))
chk = {("copy_plain", "dice"): "P1", ("copy_mirror", "dice"): "P2", ("copy_plain", "logp"): "P3", ("copy_mirror", "logp"): "P4"}
agree = [abs(A2R.set_index(["stratum", "endpoint"]).loc[k_, "estimate"] - A1.set_index("key").loc[v_, "estimate"]) < 1e-9
         for k_, v_ in chk.items() if v_ in set(A1.key)]
CHECKS["A2 the randomisation statistic equals the A1 estimate at the observed assignment"] = bool(agree) and all(agree)
print("\nRead: p_randomisation is the share of re-randomised assignments giving an estimate at least as far from 0 as the\n"
      "observed one. It conditions on the trained models (the models are not retrained under each redraw), so it tests\n"
      "whether a test image's own half explains its X1 - X2 difference beyond what any assignment would.")
print(f"\nA2 finished in {time.time() - T2:.0f}s")
done("A2")


In [ ]:
# =============================================================================
#  CELL A3  |  Metrics without the no-tumour shortcut, the crossover evaluations as
#              arms, the exposure effect by class, and how much of the available
#              headroom exposure closes. (CPU, ~1-2 min)
#  - recall_tumour3 / f1_tumour3: the three tumour classes only (a tumour called "no tumour" counts as an error). The
#    no-tumour class is separable by image size alone (manuscript Section 3.1), so 4-class macro-F1 flatters every model.
#  - Headroom: the masks of a test image and of its training twin agree at Dice ~0.95, so a model that memorised the twin
#    could gain at most (twin Dice - unexposed Dice). Needs twin_mask_dice.csv from Cell B1 (main notebook); without it,
#    that part is skipped and says so.
# =============================================================================
assert "combine" in globals(), "Run Cells A0 and A1 first."
T3 = time.time()

# ----- A3.1 the crossover evaluations as arms ---------------------------------------------------------------------
common = sorted(set(P.get("X1_flip", {})) & set(P.get("X2_flip", {})))
for name, pick1 in (("XO_unexposed", (half != 1)), ("XO_exposed", (half != 2))):
    # unexposed: half 1 -> X2, half 2 -> X1, unlinked -> X1 (as in the manuscript); exposed: half 1 -> X1, half 2 -> X2
    P[name] = {}
    for s in common:
        a, b = P["X1_flip"][s], P["X2_flip"][s]
        P[name][s] = {k: (np.where(pick1, a[k], b[k]) if np.ndim(a[k]) == 1 else np.where(pick1[:, None], a[k], b[k]))
                      for k in a}
    TASK[name] = "both"

def arm_one(arm, name):
    seeds = sorted(P.get(arm, {}))
    if not seeds:
        return None
    per = [float(metric(arm, s, name, W1, "W1")[0]) for s in seeds]
    boot = np.mean([metric(arm, s, name, W, "W") for s in seeds], 0)
    mask = tum if name == "dice" else np.ones(n, bool)
    return combine(float(np.mean(per)), boot, per, ngroups(mask))

banner("A3.1  EVERY EVALUATION, WITH AND WITHOUT THE NO-TUMOUR CLASS  (mean of seeds; group-bootstrap 95% CI)")
EVALS = [("Standard model", "full_flip"), ("Clean model", "B_flip"), ("Size-matched model", "Fm_flip"),
         ("Crossover, relatives unseen", "XO_unexposed"), ("Crossover, relatives seen", "XO_exposed"),
         ("Standard set, classification only", "full_flip_cls"), ("Clean set, classification only", "B_flip_cls"),
         ("Earlier design, clean set", "B_flip_p1"), ("EfficientNet-B0, standard", "full_flip_effb0"),
         ("EfficientNet-B0, clean", "B_flip_effb0"), ("Swin-T, standard", "full_flip_swint"), ("Swin-T, clean", "B_flip_swint")]
MET = ["macro_f1", "accuracy", "balanced_acc", "f1_tumour3", "recall_tumour3", "dice"]
rows_ = []
for lab_, arm in EVALS:
    if arm not in P:
        continue
    for m_ in MET:
        if m_ == "dice" and TASK[arm] == "cls":
            continue
        if m_ != "dice" and TASK[arm] == "seg":
            continue
        r = arm_one(arm, m_)
        rows_.append({"evaluation": lab_, "arm": arm, "metric": m_, "value": r["estimate"], "lo": r["lo"], "hi": r["hi"],
                      "boot_lo": r["boot_lo"], "boot_hi": r["boot_hi"], "per_seed": r["per_seed"]})
A3E = pd.DataFrame(rows_); A3E.to_csv(OUT / "A3_evaluations.csv", index=False)
# shown: the group-bootstrap percentile interval (bounded in [0, 1]); the t-based interval is in the CSV (lo, hi)
wide = A3E.assign(cell=lambda d: d.value.map("{:.4f}".format) + " [" + d.boot_lo.map("{:.4f}".format) + ", " + d.boot_hi.map("{:.4f}".format) + "]")
with pd.option_context("display.width", 260, "display.max_colwidth", 30):
    print(wide.pivot(index="evaluation", columns="metric", values="cell").reindex([e for e, _ in EVALS if e in set(wide.evaluation)])
          .reindex(columns=MET).to_string())
REF3 = {("full_flip", "macro_f1"): .9918, ("full_flip", "dice"): .8782, ("B_flip", "macro_f1"): .9344, ("B_flip", "dice"): .7855,
        ("XO_unexposed", "macro_f1"): .9627, ("XO_unexposed", "dice"): .8507, ("Fm_flip", "macro_f1"): .9400, ("Fm_flip", "dice"): .8069}
dd = [abs(A3E[(A3E.arm == a) & (A3E.metric == m)].value.iloc[0] - v) for (a, m), v in REF3.items()
      if ((A3E.arm == a) & (A3E.metric == m)).any()]
print(f"\nlargest difference from Table 3 of the manuscript: {max(dd):.5f}" if dd else "")
CHECKS["A3 Table 3 values reproduced (|diff| < 0.0006)"] = bool(dd) and max(dd) < 6e-4

banner("A3.2  CONTRASTS ON THE TUMOUR CLASSES ONLY  (corrected 95% CI; descriptive, not in the frozen families)")
rows_ = []
for lab_, a_, b_ in (("Standard minus clean", "full_flip", "B_flip"), ("Size-matched minus clean", "Fm_flip", "B_flip"),
                     ("Crossover: seen minus unseen (all images)", "XO_exposed", "XO_unexposed"),
                     ("Crossover unseen minus clean", "XO_unexposed", "B_flip"), ("Standard minus crossover unseen", "full_flip", "XO_unexposed")):
    for m_ in ("f1_tumour3", "recall_tumour3", "macro_f1", "dice"):
        r = arm_diff(a_, b_, m_)
        if r is not None:
            rows_.append({"contrast": lab_, "metric": m_, "estimate": r["estimate"], "lo": r["lo"], "hi": r["hi"], "p": r["p"],
                          "per_seed": r["per_seed"]})
A3C = pd.DataFrame(rows_); A3C.to_csv(OUT / "A3_tumour_contrasts.csv", index=False)
print(A3C.round(4).to_string(index=False))
print("Note: 'seen minus unseen (all images)' pools every test image; the stratum-wise estimates are the A1 rows.")

# ----- A3.3 exposure effect by class within each kind of relative -------------------------------------------------
banner("A3.3  EXPOSURE EFFECT ON DICE AND ON CORRECTNESS, BY KIND OF RELATIVE AND CLASS  (descriptive)")
def crossover_m(arm1, arm2, field, stratum, extra):
    D, seeds = xo_delta(arm1, arm2, field)
    Dm = nanmean0(D); m_ = (strat == stratum) & np.isfinite(Dm) & extra
    if m_.sum() < 2:
        return None
    est, both = half_balanced(Dm, W1, m_); boot = half_balanced(Dm, W, m_)[0]
    per = [float(half_balanced(np.nan_to_num(D[k]), W1, m_)[0][0]) for k in range(len(seeds))]
    r = combine(float(est[0]), boot, per, ngroups(m_)); r.update(images=int(m_.sum()), both_halves=both); return r
rows_ = []
for st in LINKED:
    for c in (1, 2, 3):
        for f in ("dice", "correct"):
            r = crossover_m("X1_flip", "X2_flip", f, st, lab == c)
            if r is not None:
                U = P["XO_unexposed"]; Ex = P["XO_exposed"]
                mk = (strat == st) & (lab == c)
                rows_.append({"relative": st, "class": CLASS[c], "endpoint": f, "images": r["images"], "both_halves": r["both_halves"],
                              "unexposed_mean": float(np.nanmean(nanmean0(np.stack([U[s][f] for s in U]))[mk])),
                              "exposed_mean": float(np.nanmean(nanmean0(np.stack([Ex[s][f] for s in Ex]))[mk])),
                              "effect": r["estimate"], "lo": r["lo"], "hi": r["hi"]})
A3X = pd.DataFrame(rows_); A3X.to_csv(OUT / "A3_exposure_by_class.csv", index=False)
with pd.option_context("display.max_rows", 100, "display.width", 200):
    print(A3X.round(4).to_string(index=False))

# ----- A3.4 headroom -------------------------------------------------------------------------------------------------
banner("A3.4  HOW MUCH OF THE AVAILABLE HEADROOM DOES EXPOSURE CLOSE?  (needs twin_mask_dice.csv from Cell B1)")
TW = OUT / "twin_mask_dice.csv"
if TW.exists():
    tw = pd.read_csv(TW)
    tdice = TEST.image_id.map(tw.groupby("image_id").twin_dice.max()).to_numpy(dtype=float)
    U = nanmean0(np.stack([P["XO_unexposed"][s]["dice"] for s in common]))
    Ex = nanmean0(np.stack([P["XO_exposed"][s]["dice"] for s in common]))
    rows_ = []
    for st in ("copy_plain", "copy_mirror"):
        for c in (None, 1, 2, 3):
            m_ = (strat == st) & np.isfinite(U) & np.isfinite(Ex) & np.isfinite(tdice) & (True if c is None else (lab == c))
            if m_.sum() < 3:
                continue
            u, e, t = (wmean(v, W, m_) for v in (U, Ex, tdice))
            ratio = (e - u) / (t - u)
            u0, e0, t0 = (float(wmean(v, W1, m_)[0]) for v in (U, Ex, tdice))
            lo_, hi_ = np.nanpercentile(ratio, [2.5, 97.5])
            rows_.append({"relative": st, "class": "all" if c is None else CLASS[c], "images": int(m_.sum()),
                          "unexposed_dice": u0, "exposed_dice": e0, "twin_mask_dice": t0,
                          "share_of_headroom_closed": (e0 - u0) / (t0 - u0) if t0 > u0 else np.nan, "lo": lo_, "hi": hi_})
    A3H = pd.DataFrame(rows_); A3H.to_csv(OUT / "A3_headroom.csv", index=False)
    print(A3H.round(4).to_string(index=False))
    print("share_of_headroom_closed = (exposed - unexposed) / (twin mask Dice - unexposed); 95% interval by group bootstrap\n"
          "(seed-averaged values; seed variance not included). Pooled over halves, not half-balanced: descriptive.")
else:
    print("twin_mask_dice.csv not found in brisc_round4: run Cell B1 in the main notebook (after Cells 1-7), then re-run A3.")
print(f"\nA3 finished in {time.time() - T3:.0f}s")
done("A3")


In [ ]:
# =============================================================================
#  CELL A4  |  Explanations, per image, with the corrected interval (group + seed
#              variance) for every method, class and model. (CPU, ~1-2 min)
#  Replaces the summary-level explanation numbers of the manuscript (Figure 7, Table S5), which had no corrected
#  analysis. Uses the Cell 32 tables cached in brisc_reuse.
# =============================================================================
assert "X32" in globals() and "combine" in globals(), "Run Cells A0 and A1 first."
T4 = time.time()
assert X32, "No Cell 32 tables were found in brisc_reuse: send the A0 output."

XP = {}
for tag, tab in X32.items():
    arm, seed = split_tag(tag)
    t_ = tab.copy(); t_["k"] = t_.row.map(POS)
    t_ = t_[t_.k.notna()]
    rec = {}
    for c_ in [c for c in t_.columns if c.startswith("x_")]:
        v = np.full(n, np.nan); v[t_.k.astype(int).to_numpy()] = pd.to_numeric(t_[c_], errors="coerce").to_numpy(dtype=float)
        rec[c_] = v
    XP.setdefault(arm, {})[seed] = rec
print({a: sorted(d_) for a, d_ in XP.items()})

def xmat(arm, field):
    d_ = XP.get(arm, {}); seeds = sorted(s for s in d_ if field in d_[s])
    return (np.stack([d_[s][field] for s in seeds]) if seeds else None), seeds

def one_sample(M, mask=None):
    if M is None:
        return None
    x = nanmean0(M); m_ = np.isfinite(x) & (np.ones(n, bool) if mask is None else mask)
    if m_.sum() < 2:
        return None
    per = [float(wmean(M[k], W1, m_ & np.isfinite(M[k]))[0]) for k in range(len(M))]
    r = combine(float(wmean(x, W1, m_)[0]), wmean(x, W, m_), per, ngroups(m_)); r["images"] = int(m_.sum()); return r

def xdiff(arm_a, arm_b, field, mask=None):
    Ma, sa = xmat(arm_a, field); Mb, sb = xmat(arm_b, field)
    if Ma is None or Mb is None:
        return None
    common_ = sorted(set(sa) & set(sb))
    return one_sample(Ma[[sa.index(s) for s in common_]] - Mb[[sb.index(s) for s in common_]], mask)

def xcross(field, stratum):
    M1, s1 = xmat("X1_flip_w", field); M2, s2 = xmat("X2_flip_w", field)
    if M1 is None or M2 is None:
        return None
    common_ = sorted(set(s1) & set(s2))
    D = np.stack([np.where(half == 1, M1[s1.index(s)] - M2[s2.index(s)],
                           np.where(half == 2, M2[s2.index(s)] - M1[s1.index(s)], M1[s1.index(s)] - M2[s2.index(s)]))
                  for s in common_])
    Dm = nanmean0(D); m_ = (strat == stratum) & np.isfinite(Dm)
    if m_.sum() < 2:
        return None
    est, _ = half_balanced(Dm, W1, m_); boot = half_balanced(Dm, W, m_)[0]
    per = [float(half_balanced(np.nan_to_num(D[k]), W1, m_)[0][0]) for k in range(len(common_))]
    r = combine(float(est[0]), boot, per, ngroups(m_)); r["images"] = int(m_.sum()); return r

METH = {"gc": "Grad-CAM, last stage", "gcpp": "Grad-CAM++, last stage", "gc3": "Grad-CAM, third stage", "occ": "Occlusion"}
rows_ = []
def add(block, what, r, **kw):
    if r is not None:
        rows_.append({"block": block, "quantity": what, **kw, "estimate": r["estimate"], "lo": r["lo"], "hi": r["hi"],
                      "p": r["p"], "mde": r["mde"], "per_seed": r["per_seed"], "images": r["images"]})

ARM = "B_flip_w"
for m_, nm in METH.items():
    add("H1 weight randomisation (rho, pass < 0.5)", nm, one_sample(xmat(ARM, f"x_rho_{m_}")[0]), model="clean")
    add("H2 excess mass in the BRISC mask", nm, one_sample(xmat(ARM, f"x_em_{m_}")[0], tum), model="clean")
    M_pt, _ = xmat(ARM, f"x_pt_{m_}"); M_sh, _ = xmat(ARM, "x_share")
    add("Pointing game (hit rate)", nm, one_sample(M_pt, tum), model="clean")
    if M_pt is not None and M_sh is not None and M_pt.shape == M_sh.shape:
        add("Pointing game minus chance (per-image mask share of the head)", nm, one_sample(M_pt - M_sh, tum), model="clean")
    add("Map mass outside the head", nm, one_sample(xmat(ARM, f"x_out_{m_}")[0], tum), model="clean")
    for c in (1, 2, 3):
        add("H2 excess mass by class", nm, one_sample(xmat(ARM, f"x_em_{m_}")[0], tum & (lab == c)), model="clean", cls=CLASS[c])
add("H3 tumour removal minus control removal (log p of true class)", "all classes", one_sample(xmat(ARM, "x_h3")[0], tum), model="clean")
for c in (1, 2, 3):
    add("H3 tumour removal minus control removal (log p of true class)", CLASS[c],
        one_sample(xmat(ARM, "x_h3")[0], tum & (lab == c)), model="clean", cls=CLASS[c])
for st in ("copy_plain", "copy_mirror", "patient_only", "unlinked"):
    add("H4 exposure effect on Grad-CAM excess mass", st, xcross("x_em_gc", st), model="crossover replicates")
    add("H4 exposure effect on the tumour-removal contrast", st, xcross("x_h3", st), model="crossover replicates")
add("Standard minus clean replicate", "Grad-CAM excess mass", xdiff("full_flip_w", "B_flip_w", "x_em_gc", tum), model="both")
add("Standard minus clean replicate", "Tumour-removal contrast", xdiff("full_flip_w", "B_flip_w", "x_h3", tum), model="both")
for arm_, nm in (("B_flip_p1", "earlier design"), ("B_flip_effb0", "EfficientNet-B0"), ("B_flip_swint", "Swin-T"),
                 ("full_flip_w", "our network, standard set"), ("B_flip_film_w", "our network with FiLM")):
    add("Other models: Grad-CAM excess mass", nm, one_sample(xmat(arm_, "x_em_gc")[0], tum), model=arm_)
    add("Other models: tumour-removal contrast", nm, one_sample(xmat(arm_, "x_h3")[0], tum), model=arm_)
A4 = pd.DataFrame(rows_); A4.to_csv(OUT / "A4_explanations_corrected.csv", index=False)
with pd.option_context("display.max_rows", 300, "display.width", 240, "display.max_colwidth", 60):
    print(A4.drop(columns=["model"]).round(4).to_string(index=False))
REF4 = {("H2 excess mass in the BRISC mask", "Grad-CAM, last stage"): .0311,
        ("H3 tumour removal minus control removal (log p of true class)", "all classes"): -1.5826}
dd = [abs(A4[(A4.block == b) & (A4.quantity == q)].estimate.iloc[0] - v) for (b, q), v in REF4.items()
      if ((A4.block == b) & (A4.quantity == q)).any()]
CHECKS["A4 the pre-specified H2 and H3 estimates are reproduced (|diff| < 0.001)"] = bool(dd) and max(dd) < 1e-3
print(f"\nA4 finished in {time.time() - T4:.0f}s")
done("A4")


In [ ]:
# =============================================================================
#  CELL A5  |  Orientation of every traced image relative to its figshare source
#              slice (8 orientations were searched in Cell 5). (seconds)
#  If any BRISC image is a ROTATED copy of its source, two BRISC images could also be rotated copies of each other,
#  which the train-test search (identity and left-right mirror only) cannot see. Cell B1 then checks that directly.
# =============================================================================
assert "man" in globals(), "Run Cell A0 first."
banner("A5  ORIENTATION RELATIVE TO THE FIGSHARE SOURCE (traced images)")
if "fig_variant" in man and "traced" in man:
    tr_ = man[man.traced].copy(); tr_["fig_variant"] = tr_.fig_variant.fillna("?").astype(str)
    A5 = pd.crosstab([tr_.split, tr_.label], tr_.fig_variant, margins=True)
    print(A5.to_string()); A5.to_csv(OUT / "A5_orientation_vs_source.csv")
    rot = tr_.fig_variant.str.startswith("rot")
    print(f"\ntraced images: {len(tr_)}; stored mirrored: {int(tr_.fig_variant.str.contains('mirror').sum())}; "
          f"stored rotated (90/180/270, with or without mirror): {int(rot.sum())}")
    if rot.any():
        print("Rotated copies exist relative to the source: run Cell B1 (8-orientation train-test search).")
    CHECKS["A5 orientation table written"] = True
else:
    print("manifest has no fig_variant/traced columns"); CHECKS["A5 orientation table written"] = False
done("A5")

# =============================================================================
#  CELL A9  |  Pack every output of this notebook into one zip and download it
# =============================================================================
banner("A9  PACKAGE")
summary = {"generated": time.strftime("%Y-%m-%d %H:%M:%S"), "n_boot": N_BOOT, "boot_seed": BOOT_SEED,
           "checks": CHECKS, "arms": {a: sorted(d_) for a, d_ in P.items()},
           "explanation_tables": sorted(X32), "files": sorted(p.name for p in OUT.iterdir() if p.is_file())}
(OUT / "A9_summary.json").write_text(json.dumps(summary, indent=2, default=str))
zpath = OUT / "brisc_round4_results.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p_ in sorted(OUT.iterdir()):
        if p_.is_file() and p_.name != zpath.name:
            z.write(p_, p_.name)
print(f"{zpath}  ({zpath.stat().st_size / 1e6:.1f} MB)")
print("FAILED CHECKS:", [k for k, v in CHECKS.items() if not v] or "none")
if not MOCK:
    from google.colab import files
    files.download(str(zpath))


## Cell B1 — for the MAIN notebook (`BRISC_contamination_controlled_multitask_v3.ipynb`)

Open the main notebook in a fresh session, run **Cells 1–7** (they reload the pixel searches from `MyDrive/brisc_reuse`; your last run did this in a few minutes), then paste this cell after Cell 7 and run it (~2–5 min).
It writes `twin_mask_dice.csv`, `rotation_twins.csv` and `B1_summary.json` to `MyDrive/brisc_round4/`. Afterwards re-run **A3** here (or send me the three files).
In this notebook the cell only prints that it was skipped.

In [ ]:
# =============================================================================
#  CELL B1  |  (main notebook, run right after Cells 1-7; Cells 2-7 reload their
#              searches from Drive) Per-image Dice between each test mask and its
#              training twin's mask; and a train-test copy search in all 8
#              orientations (rotations were never searched between BRISC images).
#              CPU; ~3-6 min after Cells 1-7.
# =============================================================================
_need = ("mk_ev", "MKc", "align", "get_img", "ncc_best", "grad_vec", "VEC2", "d4", "test_idx", "train_idx", "COPY_NCC",
         "df", "DRIVE_RUNS", "WORK", "finish")
if not all(v in globals() for v in _need):
    print("Cell B1 belongs in the MAIN notebook, after Cells 1-7. Skipped here.")
else:
  exec_b1 = True
  OUT4 = (DRIVE_RUNS.parent if DRIVE_RUNS is not None else WORK) / "brisc_round4"; OUT4.mkdir(parents=True, exist_ok=True)
  checks, notes = {}, []
  K_ROT = 10                                   # coarse candidates checked at full resolution per test image and orientation

  # ----- B1.1 per-image twin-mask Dice (the same pairs and alignment as Cell 7.5) ---------------------------------------
  banner("B1.1  DICE BETWEEN EACH TEST MASK AND ITS TRAINING TWIN'S MASK, PER IMAGE")
  rec = []
  for _, w_ in mk_ev.iterrows():
      al = align(np.asarray(MKc[int(w_.twin_row)]), int(w_.dx), int(w_.dy))
      al = al[:, ::-1] if w_.mirrored else al
      a_, b_ = MKc[int(w_.row)] > 0, al > 0
      if a_.any() and b_.any():
          rec.append({"image_id": df.image_id.iat[int(w_.row)], "twin_id": df.image_id.iat[int(w_.twin_row)],
                      "mirrored": bool(w_.mirrored), "ncc": float(w_.ncc), "twin_dice": 2 * (a_ & b_).sum() / (a_.sum() + b_.sum())})
  TWD = pd.DataFrame(rec)
  TWD.to_csv(OUT4 / "twin_mask_dice.csv", index=False)
  print(TWD.groupby("mirrored").twin_dice.agg(["size", "median"]).round(3).to_string())
  checks["B1 per-image twin Dice reproduces Cell 7.5 (534 pairs, median 0.946)"] = (
      len(TWD) == 534 and abs(TWD.twin_dice.median() - 0.946) < 1e-3)

  # ----- B1.2 eight orientations ---------------------------------------------------------------------------------------
  banner(f"B1.2  TRAIN-TEST COPY SEARCH IN THE 6 ROTATED ORIENTATIONS (identity and mirror were searched in Cells 6-7)")
  t0 = time.time(); TRN = np.asarray(train_idx); hits = []
  for i in np.asarray(test_idx):
      a = get_img(int(i))
      for v in range(2, 8):                                             # rot90, rot90+mirror, ..., rot270+mirror
          av = np.ascontiguousarray(d4(a, v)).astype(np.float32)
          sc = VEC2[TRN] @ grad_vec(av)
          for o in np.argsort(-sc)[:K_ROT]:
              ncc, dx, dy = ncc_best(av, get_img(int(TRN[o])))
              if ncc >= COPY_NCC:
                  hits.append({"row": int(i), "image_id": df.image_id.iat[int(i)], "twin_id": df.image_id.iat[int(TRN[o])],
                               "orientation": ["identity", "mirror", "rot90", "rot90+mirror", "rot180", "rot180+mirror",
                                               "rot270", "rot270+mirror"][v], "ncc": ncc,
                               "stratum": df.stratum.iat[int(i)], "label": df.label.iat[int(i)]})
  ROT = pd.DataFrame(hits, columns=["row", "image_id", "twin_id", "orientation", "ncc", "stratum", "label"])
  ROT.to_csv(OUT4 / "rotation_twins.csv", index=False)
  print(f"{len(test_idx)} test images x 6 orientations x {K_ROT} candidates in {time.time() - t0:.0f}s")
  new_ = ROT[~ROT.stratum.isin(["copy_plain", "copy_mirror"])]
  print(f"test images with a ROTATED near-identical training image: {ROT.row.nunique()}; "
        f"of these, not already counted as copies: {new_.row.nunique()} {new_.drop_duplicates('row').stratum.value_counts().to_dict()}")
  if len(new_):
      print(new_.sort_values("ncc", ascending=False).head(20).to_string(index=False))
  checks["B1 every test image was searched in the 6 rotated orientations"] = True
  (OUT4 / "B1_summary.json").write_text(json.dumps({"twin_pairs": len(TWD), "twin_dice_median": float(TWD.twin_dice.median()),
                                                     "rotated_twins_images": int(ROT.row.nunique()),
                                                     "rotated_twins_new": int(new_.row.nunique()),
                                                     "by_stratum_new": new_.drop_duplicates("row").stratum.value_counts().to_dict()}, indent=2))
  print(f"saved twin_mask_dice.csv, rotation_twins.csv, B1_summary.json in {OUT4}")
  finish("B1", checks, notes)
